# 🚀 HUẤN LUYỆN YOLO26n TRÊN GOOGLE COLAB
### Dành cho bộ dữ liệu Traffic Tracking (6 Classes: person, bicycle, car, motorbike, bus, truck)

* **Dataset**: 100 ảnh đã gán nhãn & kiểm duyệt từ CVAT (chia tỉ lệ 7:2:1 -> 70 train, 20 val, 10 test).
* **Mục tiêu**: Fine-tuning mô hình YOLO26n (Ultralytics), đánh giá mAP, trực quan hóa kết quả và xuất sang OpenVINO INT8 / FP16 & ONNX để tích hợp vào hệ thống UKF-DeepSORT.


In [ ]:
# 1. Kiểm tra môi trường GPU
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


## 2. Cài đặt các thư viện cần thiết
Cài đặt `ultralytics` cho YOLO, `openvino` để lượng tử hóa và xuất mô hình tối ưu Intel Edge/CPU, và `onnx`.


In [ ]:
!pip install -q "ultralytics>=8.3.0" openvino nncf onnx


## 3. Tải và Giải nén Dữ liệu
Bạn có 2 cách đưa dữ liệu lên Google Colab:
- **Cách 1 (Nhanh nhất)**: Tải trực tiếp file `yolo26n_colab_package.zip` từ máy tính lên thư mục gốc của Colab (biểu tượng thư mục tab bên trái), sau đó chạy lệnh giải nén bên dưới.
- **Cách 2**: Đặt file ZIP trên Google Drive và kết nối Drive.


In [ ]:
import os
import glob
import shutil
from pathlib import Path

zip_path = "yolo26n_colab_package.zip"

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Tìm thấy {zip_path} (Dung lượng: {size_mb:.2f} MB)")
    if size_mb < 60.0:
        print(f"⚠️ CẢNH BÁO: File mới tải lên được {size_mb:.2f} MB / 62.8 MB (đang upload dở dang).")
        print("👉 Hãy nhìn vào vòng tròn tiến trình bên cạnh file ở thanh bên trái (Files).")
        print("👉 Vui lòng đợi vòng tròn xoay hết 100% rồi ấn chạy lại ô này!")
    else:
        print("📦 Đang giải nén file package...")
        !unzip -q -o yolo26n_colab_package.zip
        print("✅ Giải nén hoàn tất! Các thư mục sẵn sàng:")
        !ls -d */
elif os.path.exists("yolo26n_dataset"):
    print("✅ Thư mục yolo26n_dataset đã có sẵn và sẵn sàng!")
else:
    # Tùy chọn 2: Tìm trong Google Drive nếu đã tải lên Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_zips = glob.glob('/content/drive/MyDrive/**/yolo26n_colab_package.zip', recursive=True)
        if drive_zips:
            print(f"Tìm thấy file trên Google Drive: {drive_zips[0]}")
            !unzip -q -o "{drive_zips[0]}" -d /content/
            print("✅ Giải nén từ Google Drive thành công!")
        else:
            print("⚠️ Không tìm thấy file trong Drive. Vui lòng kéo thả yolo26n_colab_package.zip vào Colab!")
    except Exception as e:
        print("Vui lòng tải file yolo26n_colab_package.zip lên tab Files bên trái:", e)


## 4. Kiểm tra Dữ liệu & Trực quan hóa Nhãn Mẫu
Kiểm tra số lượng ảnh trong từng tập `train` (70), `val` (20), `test` (10) và hiển thị bounding box ảnh mẫu.


In [ ]:
import cv2
import yaml
import glob
import matplotlib.pyplot as plt

data_yaml_path = "yolo26n_dataset/data.yaml"
with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print("Cấu hình data.yaml:")
print(data_cfg)

# Đếm số lượng ảnh
for split in ['train', 'val', 'test']:
    imgs = glob.glob(f"yolo26n_dataset/images/{split}/*.jpg")
    lbls = glob.glob(f"yolo26n_dataset/labels/{split}/*.txt")
    print(f"Tập {split.upper():5s}: {len(imgs):2d} ảnh | {len(lbls):2d} file nhãn")

# Hiển thị thử 1 ảnh mẫu tập train kèm bounding boxes
sample_img_path = sorted(glob.glob("yolo26n_dataset/images/train/*.jpg"))[0]
sample_txt_path = sample_img_path.replace("images", "labels").replace(".jpg", ".txt")

img = cv2.imread(sample_img_path)
h, w, _ = img.shape
names = data_cfg['names']

if os.path.exists(sample_txt_path):
    with open(sample_txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, names[cls_id], (x1, max(y1 - 5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"Mẫu ảnh gán nhãn: {os.path.basename(sample_img_path)}")
plt.axis("off")
plt.show()


## 5. Khởi tạo và Huấn luyện YOLO26n
* **Trọng số khởi tạo**: `weights/yolo26n.pt` hoặc `yolo26n.pt`
* **Siêu tham số khuyến nghị cho 100 ảnh**:
  - `epochs=60` hoặc `100` (được bảo vệ bởi `patience=20` Early Stopping tránh lãng phí thời gian)
  - `batch=16`
  - `imgsz=640`
  - Data Augmentation: Mosaic, Mixup, Flip, HSV giúp mô hình học khái quát tốt hơn trên lượng mẫu nhỏ, chống overfitting.


In [ ]:
from ultralytics import YOLO
import os

# Khởi tạo mô hình từ trọng số pretrained
weights_file = "weights/yolo26n.pt" if os.path.exists("weights/yolo26n.pt") else "yolo26n.pt"
print(f"Đang tải mô hình từ: {weights_file}")
model = YOLO(weights_file)

# Tiến hành huấn luyện (lưu kết quả trực tiếp)
results = model.train(
    data="yolo26n_dataset/data.yaml",
    epochs=60,
    batch=16,
    imgsz=640,
    patience=20,
    save=True,
    save_period=10,
    name="yolo26n_traffic",
    exist_ok=True,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)

train_save_dir = str(results.save_dir)
print(f"\n🎉 Huấn luyện thành công! Thư mục lưu kết quả: {train_save_dir}")


## 6. Đánh giá Mô hình trên Validation & Test Sets
Đánh giá độ chính xác tổng thể (mAP@0.5, mAP@0.5:0.95, Precision, Recall) và hiển thị Confusion Matrix.


In [ ]:
from IPython.display import Image, display
import glob, os

# 1. Đánh giá tập Validation
print("--- KẾT QUẢ ĐÁNH GIÁ TẬP VALIDATION ---")
val_metrics = model.val(data="yolo26n_dataset/data.yaml", split="val")
print(f"Validation mAP50:    {val_metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {val_metrics.box.map:.4f}")

# 2. Đánh giá tập Test độc lập
print("\n--- KẾT QUẢ ĐÁNH GIÁ TẬP TEST ĐỘC LẬP ---")
test_metrics = model.val(data="yolo26n_dataset/data.yaml", split="test")
print(f"Test mAP50:    {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")

# 3. Tự động tìm và hiển thị Confusion Matrix và biểu đồ kết quả
cm_files = glob.glob("runs/**/confusion_matrix.png", recursive=True)
if cm_files:
    print("\nConfusion Matrix:")
    display(Image(cm_files[-1], width=700))

res_files = glob.glob("runs/**/results.png", recursive=True)
if res_files:
    print("\nBiểu đồ Loss và mAP qua các epoch:")
    display(Image(res_files[-1], width=800))


## 7. Trực quan hóa Dự đoán trên Tập Test (Visual Inference)
Chạy thử nghiệm mô hình tốt nhất (`best.pt`) trên các ảnh trong tập `test` để kiểm tra trực quan.


In [ ]:
import glob, os
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Tự động dò tìm file best.pt mới nhất trong thư mục runs/
found_weights = [str(p) for p in Path('runs').rglob('best.pt')]

if not found_weights:
    print("⚠️ CHƯA TÌM THẤY file best.pt!")
    print("👉 Nguyên nhân: Ô số 5 (Huấn luyện) chưa được chạy hoặc chưa chạy xong.")
    print("👉 Hãy cuộn lên Ô số 5 và bấm chạy (chờ khoảng 1-2 phút) rồi chạy lại ô này nhé!")
else:
    best_model_path = found_weights[-1]
    print(f"✅ Tải mô hình tốt nhất từ: {best_model_path}")
    best_model = YOLO(best_model_path)

    test_images = sorted(glob.glob("yolo26n_dataset/images/test/*.jpg"))[:4]
    plt.figure(figsize=(16, 10))
    for i, img_p in enumerate(test_images):
        res = best_model.predict(img_p, conf=0.25, imgsz=640)[0]
        plotted = res.plot()
        plt.subplot(2, 2, i + 1)
        plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        plt.title(f"Dự đoán: {os.path.basename(img_p)}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 8. Xuất Mô hình sang OpenVINO & ONNX
Xuất mô hình đã fine-tune sang:
1. **ONNX**
2. **OpenVINO FP16**
3. **OpenVINO INT8 (Post-Training Quantization)**: Tối ưu hoá FPS cực mạnh trên Intel CPU, phù hợp tiêu chuẩn của repo benchmark.


In [ ]:
from pathlib import Path
from ultralytics import YOLO

found_weights = [str(p) for p in Path('runs').rglob('best.pt')]
if found_weights:
    best_model = YOLO(found_weights[-1])
    print("1. Xuất mô hình ONNX...")
    best_model.export(format="onnx", imgsz=640)

    print("2. Xuất mô hình OpenVINO FP16...")
    best_model.export(format="openvino", imgsz=640, half=True)

    print("3. Xuất mô hình OpenVINO INT8 PTQ...")
    best_model.export(
        format="openvino",
        int8=True,
        data="yolo26n_dataset/data.yaml",
        imgsz=640
    )
    print("✅ Xuất toàn bộ định dạng thành công!")
else:
    print("⚠️ Không tìm thấy file best.pt để xuất mô hình. Hãy chạy Ô số 5 trước!")


## 9. Đóng gói Kết quả & Tải về Máy tính
Nén thư mục `best.pt` cùng các mô hình OpenVINO / ONNX thành file ZIP để tải về máy hoặc lưu vào Google Drive.


In [ ]:
import shutil
import os
from pathlib import Path
from google.colab import files

output_dir = "yolo26n_trained_results"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# 1. Copy trọng số best.pt
found_weights = [p for p in Path("runs").rglob("best.pt")]
if found_weights:
    target_pt = f"{output_dir}/yolo26n_traffic_best.pt"
    shutil.copy(str(found_weights[-1]), target_pt)
    print(f"✅ Đã copy weights: {found_weights[-1]} -> {target_pt}")
else:
    print("⚠️ Không tìm thấy file best.pt trong runs/")

# 2. Copy OpenVINO models
for item in Path("runs").rglob("*openvino*"):
    if item.name in ["weights", "train"]:
        continue
    dest = os.path.join(output_dir, item.name)
    if item.is_dir():
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(str(item), dest)
        print(f"✅ Đã copy thư mục OpenVINO: {item.name}")
    else:
        shutil.copy(str(item), dest)

# 3. Copy ONNX models
for item in Path("runs").rglob("*.onnx"):
    dest = os.path.join(output_dir, item.name)
    shutil.copy(str(item), dest)
    print(f"✅ Đã copy ONNX: {item.name}")

# 4. Nén kết quả
shutil.make_archive("yolo26n_trained_results", "zip", output_dir)
zip_size = os.path.getsize("yolo26n_trained_results.zip") / (1024 * 1024)
print(f"📦 File yolo26n_trained_results.zip đã sẵn sàng! Dung lượng: {zip_size:.2f} MB")

try:
    files.download("yolo26n_trained_results.zip")
except Exception as e:
    print("Tải file thủ công từ tab Files bên trái nếu không tự động tải:", e)
